# SkinDisease Local Model Training on Colab GPU (Enhanced)

Run the 8 sections in order: environment setup -> mount Drive -> dataset check -> training config -> training -> evaluation -> export artifacts -> download/sync.


In [ ]:
# 1) Environment Setup
import os
import sys
import json
import subprocess
from pathlib import Path

print('Python:', sys.version)
print('Executable:', sys.executable)
subprocess.run(['nvidia-smi'], check=False)


In [ ]:
# 2) Mount Google Drive and Set Project Path
from google.colab import drive
drive.mount('/content/drive')

# 原本的云盘路径
DRIVE_PROJECT_ROOT = '/content/drive/Othercomputers/MyComputer/Code/'

# 【新增】：将代码拷贝到本地高速固态硬盘
!cp -r "$DRIVE_PROJECT_ROOT" /content/Code/

# 【修改】：将工作目录切换到本地
os.chdir('/content/Code')
print('Working dir:', os.getcwd())

import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])

In [ ]:
# 3) Dataset Validation (22 classes, train/test consistency, no empty folders)
DATASET_ROOT = '/content/Dataset/archive/SkinDisease'  # TODO: change to your dataset path
EXPECTED_CLASSES = 22
EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def count_split(root, split):
    split_dir = Path(root) / split
    if not split_dir.exists():
        raise FileNotFoundError(f'missing split: {split_dir}')
    counts = {}
    for d in sorted([p for p in split_dir.iterdir() if p.is_dir()]):
        cnt = sum(1 for p in d.rglob('*') if p.suffix.lower() in EXTS)
        counts[d.name] = cnt
    return counts

train_counts = count_split(DATASET_ROOT, 'train')
test_counts = count_split(DATASET_ROOT, 'test')
train_labels, test_labels = set(train_counts.keys()), set(test_counts.keys())
if train_labels != test_labels:
    raise ValueError(f'label mismatch. missing_in_test={sorted(train_labels-test_labels)}, missing_in_train={sorted(test_labels-train_labels)}')
if len(train_labels) != EXPECTED_CLASSES:
    raise ValueError(f'expected {EXPECTED_CLASSES} classes, got {len(train_labels)}')

empty_train = [k for k, v in train_counts.items() if v == 0]
empty_test = [k for k, v in test_counts.items() if v == 0]
if empty_train or empty_test:
    raise ValueError(f'empty class dirs found: train={empty_train}, test={empty_test}')

print('Dataset check passed.')
print('Class count:', len(train_labels))
print('\nClass distribution (train/test):')
print(f"{'label':28s} {'train':>8s} {'test':>8s}")
print('-' * 48)
for label in sorted(train_labels):
    print(f"{label:28s} {train_counts[label]:8d} {test_counts[label]:8d}")


In [ ]:
# 4) Unified Multi-Model Training Config (Fair Comparison)
ARTIFACTS_DIR = '/content/skin_disease_artifacts_colab'
ARCH_LIST = ['mobilenet_v3_small', 'resnet18', 'efficientnet_b0']

COMMON_CFG = {
    'epochs': 30,
    'batch_size': 64,
    'lr': 3e-4,
    'weight_decay': 1e-4,
    'num_workers': 2,
    'freeze_epochs': 5,
    'pretrained': True,
    'imbalance_strategy': 'class_weight',  # class_weight | focal
    'focal_gamma': 2.0,
    'early_stop_patience': 6,
    'use_amp': True,
    'use_weighted_sampler': False,
    'seed': 42,
    'bench_warmup': 10,
    'bench_repeats': 80,
}

print('ARTIFACTS_DIR =', ARTIFACTS_DIR)
print('ARCH_LIST =', ARCH_LIST)
print('COMMON_CFG =', COMMON_CFG)



In [ ]:
# 5) Train All Models + Evaluate + Build Summary Table
import csv

summary_rows = []
artifacts_root = Path(ARTIFACTS_DIR)
artifacts_root.mkdir(parents=True, exist_ok=True)

for arch in ARCH_LIST:
    print('')
    print('=' * 80)
    print(f'[Train] arch={arch}')
    arch_dir = artifacts_root / arch
    arch_dir.mkdir(parents=True, exist_ok=True)

    train_cmd = [
        sys.executable, 'scripts/train_local_model.py',
        '--dataset-root', DATASET_ROOT,
        '--artifacts-dir', str(arch_dir),
        '--arch', arch,
        '--epochs', str(COMMON_CFG['epochs']),
        '--batch-size', str(COMMON_CFG['batch_size']),
        '--lr', str(COMMON_CFG['lr']),
        '--weight-decay', str(COMMON_CFG['weight_decay']),
        '--num-workers', str(COMMON_CFG['num_workers']),
        '--freeze-epochs', str(COMMON_CFG['freeze_epochs']),
        '--imbalance-strategy', COMMON_CFG['imbalance_strategy'],
        '--focal-gamma', str(COMMON_CFG['focal_gamma']),
        '--early-stop-patience', str(COMMON_CFG['early_stop_patience']),
        '--expected-num-classes', str(EXPECTED_CLASSES),
        '--seed', str(COMMON_CFG['seed']),
        '--bench-warmup', str(COMMON_CFG['bench_warmup']),
        '--bench-repeats', str(COMMON_CFG['bench_repeats']),
    ]
    train_cmd += ['--pretrained'] if COMMON_CFG['pretrained'] else ['--no-pretrained']
    train_cmd += ['--use-amp'] if COMMON_CFG['use_amp'] else ['--no-use-amp']
    if COMMON_CFG['use_weighted_sampler']:
        train_cmd.append('--use-weighted-sampler')

    print('Train cmd:', ' '.join(train_cmd))
    subprocess.check_call(train_cmd)

    eval_output = arch_dir / 'local_eval_report.json'
    eval_cmd = [
        sys.executable, 'scripts/evaluate_local_methods.py',
        '--dataset-root', DATASET_ROOT,
        '--artifacts-dir', str(arch_dir),
        '--max-per-class', '40',
        '--symptom-mode', 'label_hint',
        '--output', str(eval_output),
    ]
    subprocess.check_call(eval_cmd)

    metrics_path = arch_dir / 'metrics.json'
    with metrics_path.open('r', encoding='utf-8') as f:
        metrics = json.load(f)

    row = {
        'arch': arch,
        'top1': float(metrics.get('final_eval', {}).get('top1_acc', 0.0)),
        'top3': float(metrics.get('final_eval', {}).get('top3_acc', 0.0)),
        'macro_f1': float(metrics.get('final_eval', {}).get('macro_f1', 0.0)),
        'model_params': int(metrics.get('model_params', 0)),
        'model_size_mb': float(metrics.get('model_size_mb', 0.0)),
        'inference_ms_per_image': float(metrics.get('inference_ms_per_image', 0.0)),
        'best_epoch': int(metrics.get('best_epoch', 0)),
        'best_macro_f1': float(metrics.get('best_macro_f1', 0.0)),
        'train_seconds': float(metrics.get('train_seconds', 0.0)),
        'metrics_path': str(metrics_path),
    }
    summary_rows.append(row)

    print(
        f"[Done] {arch}: top1={row['top1']:.4f}, macro_f1={row['macro_f1']:.4f}, "
        f"size={row['model_size_mb']:.2f}MB, infer={row['inference_ms_per_image']:.2f}ms"
    )

compare_json = artifacts_root / 'compare_summary.json'
with compare_json.open('w', encoding='utf-8') as f:
    json.dump({'configs': COMMON_CFG, 'results': summary_rows}, f, ensure_ascii=False, indent=2)

compare_csv = artifacts_root / 'compare_summary.csv'
fieldnames = [
    'arch', 'top1', 'top3', 'macro_f1', 'model_params', 'model_size_mb', 'inference_ms_per_image',
    'best_epoch', 'best_macro_f1', 'train_seconds', 'metrics_path'
]
with compare_csv.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(summary_rows)

print('')
print('Saved comparison files:')
print('-', compare_json)
print('-', compare_csv)



In [ ]:
# 6) Plot Model Comparison Figures (Paper-Ready)
import matplotlib.pyplot as plt
import numpy as np

artifacts_root = Path(ARTIFACTS_DIR)
compare_json = artifacts_root / 'compare_summary.json'
with compare_json.open('r', encoding='utf-8') as f:
    summary = json.load(f)

rows = summary.get('results', [])
if not rows:
    raise RuntimeError('No comparison rows found. Please run cell 5 first.')

print('--- Comparison Summary ---')
for r in rows:
    print(
        f"{r['arch']}: top1={r['top1']:.4f}, top3={r['top3']:.4f}, macro_f1={r['macro_f1']:.4f}, "
        f"params={r['model_params']}, size={r['model_size_mb']:.2f}MB, infer={r['inference_ms_per_image']:.2f}ms"
    )

arches = [r['arch'] for r in rows]
top1_vals = [float(r['top1']) for r in rows]
macro_vals = [float(r['macro_f1']) for r in rows]
size_vals = [float(r['model_size_mb']) for r in rows]
lat_vals = [float(r['inference_ms_per_image']) for r in rows]

x = np.arange(len(arches))
width = 0.35

# Figure 1: Top-1 and Macro-F1
fig, ax = plt.subplots(figsize=(9, 5.2))
ax.bar(x - width / 2, top1_vals, width, label='Top-1', color='#1f77b4')
ax.bar(x + width / 2, macro_vals, width, label='Macro-F1', color='#ff7f0e')
ax.set_xticks(x)
ax.set_xticklabels(arches)
ax.set_ylim(0, 1.0)
ax.set_ylabel('Score')
ax.set_title('Model Comparison: Top-1 vs Macro-F1')
ax.grid(axis='y', alpha=0.3)
ax.legend()
for xi, v in zip(x - width / 2, top1_vals):
    ax.text(xi, v + 0.015, f'{v:.3f}', ha='center', va='bottom', fontsize=9)
for xi, v in zip(x + width / 2, macro_vals):
    ax.text(xi, v + 0.015, f'{v:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
fig1_path = artifacts_root / 'compare_top1_macrof1.png'
fig.savefig(fig1_path, dpi=170, bbox_inches='tight')
plt.show()
print('Saved plot:', fig1_path)

# Figure 2: model size and inference latency
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(arches, size_vals, color='#2ca02c')
axes[0].set_title('Model Size (MB)')
axes[0].set_ylabel('MB')
axes[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(size_vals):
    axes[0].text(i, v + (max(size_vals) * 0.02 if max(size_vals) > 0 else 0.02), f'{v:.2f}', ha='center', va='bottom', fontsize=9)

axes[1].bar(arches, lat_vals, color='#9467bd')
axes[1].set_title('Inference Time per Image (ms)')
axes[1].set_ylabel('ms')
axes[1].grid(axis='y', alpha=0.3)
for i, v in enumerate(lat_vals):
    axes[1].text(i, v + (max(lat_vals) * 0.02 if max(lat_vals) > 0 else 0.02), f'{v:.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
fig2_path = artifacts_root / 'compare_size_latency.png'
fig.savefig(fig2_path, dpi=170, bbox_inches='tight')
plt.show()
print('Saved plot:', fig2_path)



In [ ]:
# 7) Export Artifacts (Per-Model + Comparison Summary)
import shutil
import zipfile

artifacts_root = Path(ARTIFACTS_DIR)
export_dir = Path('/content/drive/MyDrive/skin_disease_artifacts_export_multi')
if export_dir.exists():
    shutil.rmtree(export_dir)
export_dir.mkdir(parents=True, exist_ok=True)

# Export summary files
summary_files = [
    'compare_summary.json',
    'compare_summary.csv',
    'compare_top1_macrof1.png',
    'compare_size_latency.png',
]
for name in summary_files:
    src = artifacts_root / name
    if src.exists():
        shutil.copy2(src, export_dir / name)

# Export key artifacts for each model
needed = ['local_model.pkl', 'label_map.json', 'metrics.json', 'train_manifest.csv', 'test_manifest.csv', 'local_eval_report.json']
for arch in ARCH_LIST:
    src_dir = artifacts_root / arch
    dst_dir = export_dir / arch
    dst_dir.mkdir(parents=True, exist_ok=True)
    for name in needed:
        src = src_dir / name
        if src.exists():
            shutil.copy2(src, dst_dir / name)

zip_local = Path('/content/skin_disease_artifacts_export_multi.zip')
with zipfile.ZipFile(zip_local, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for pth in sorted(export_dir.rglob('*')):
        if pth.is_file():
            zf.write(pth, arcname=str(pth.relative_to(export_dir)))

zip_drive = Path('/content/drive/MyDrive/skin_disease_artifacts_export_multi.zip')
shutil.copy2(zip_local, zip_drive)

print('Export dir:', export_dir)
#print('Zip local:', zip_local)
print('Zip drive:', zip_drive)



In [ ]:
# 8) Download / Sync
# A) Download the zip file to your local machine
from google.colab import files
files.download('/content/skin_disease_artifacts_export_multi.zip')

# B) You can also download directly from Google Drive:
# /content/drive/MyDrive/skin_disease_artifacts_export_multi.zip

